<a href="https://colab.research.google.com/github/krishna2426/subway-champ/blob/main/subway_surfers_1stdraft.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:

!pip install stable-baselines3[extra] gymnasium pygame
!pip install moviepy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.0/188.0 kB 7.8 MB/s eta 0:00:00


In [4]:
import gymnasium as gym
from gymnasium import spaces
import numpy as np

class SimpleSurferEnv(gym.Env):
    def __init__(self):
        super(SimpleSurferEnv, self).__init__()
        # Actions: 0=Left, 1=Stay/NoOp, 2=Right (Simplified for now)
        self.action_space = spaces.Discrete(3)

        # Observation: The game state (Player Lane, Obstacle Lane, Distance)
        # We use a simple array instead of pixels for the "Starting" phase to make training instant
        self.observation_space = spaces.Box(low=0, high=2, shape=(3,), dtype=np.float32)

        self.player_lane = 1 # Start in middle (0, 1, 2)
        self.obstacle_lane = 0
        self.score = 0

    def reset(self, seed=None):
        self.player_lane = 1
        self.obstacle_lane = np.random.randint(0, 3)
        return np.array([self.player_lane, self.obstacle_lane, 0], dtype=np.float32), {}

    def step(self, action):
        # 1. Apply Move
        if action == 0: self.player_lane = max(0, self.player_lane - 1) # Move Left
        if action == 2: self.player_lane = min(2, self.player_lane + 1) # Move Right

        # 2. Check Collision (Did we hit the obstacle?)
        # In this simple version, obstacle is always "approaching"
        terminated = False
        reward = 1 # Survival reward

        if self.player_lane == self.obstacle_lane:
            reward = -10 # Crash!
            terminated = True

        # 3. New Obstacle
        self.obstacle_lane = np.random.randint(0, 3)

        return np.array([self.player_lane, self.obstacle_lane, 0], dtype=np.float32), reward, terminated, False, {}

# Register the environment so we can use it
env = SimpleSurferEnv()

In [7]:
from stable_baselines3 import PPO

# 1. Initialize the AI Agent
model = PPO("MlpPolicy", env, verbose=1)

# 2. Train the Agent
print("Training started...")
model.learn(total_timesteps=10000) # The AI plays 10,000 frames instantly
print("Training finished!")

Using cuda device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
Training started...
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 3.22     |
|    ep_rew_mean     | -7.78    |
| time/              |          |
|    fps             | 819      |
|    iterations      | 1        |
|    time_elapsed    | 2        |
|    total_timesteps | 2048     |
---------------------------------
----------------------------------------
| rollout/                |            |
|    ep_len_mean          | 3.64       |
|    ep_rew_mean          | -7.36      |
| time/                   |            |
|    fps                  | 644        |
|    iterations           | 2          |
|    time_elapsed         | 6          |
|    total_timesteps      | 4096       |
| train/                  |            |
|    approx_kl            | 0.01644003 |
|    clip_fraction        | 0.223      |
|    clip_range           | 0.2        |
|    entropy

In [6]:
obs, _ = env.reset()
for i in range(10):
    action, _states = model.predict(obs)
    obs, reward, done, _, _ = env.step(action)
    print(f"Step {i}: Action={action}, Resulting Lane={obs[0]}, Game Over?={done}")

Step 0: Action=2, Resulting Lane=2.0, Game Over?=False
Step 1: Action=0, Resulting Lane=1.0, Game Over?=False
Step 2: Action=2, Resulting Lane=2.0, Game Over?=False
Step 3: Action=0, Resulting Lane=1.0, Game Over?=False
Step 4: Action=0, Resulting Lane=0.0, Game Over?=False
Step 5: Action=2, Resulting Lane=1.0, Game Over?=False
Step 6: Action=0, Resulting Lane=0.0, Game Over?=False
Step 7: Action=0, Resulting Lane=0.0, Game Over?=False
Step 8: Action=0, Resulting Lane=0.0, Game Over?=False
Step 9: Action=2, Resulting Lane=1.0, Game Over?=False
